## Install pyspark and download the 7 log files

In [ ]:
# Install Java and Spark dependencies
!apt-get update
!apt-get install -y openjdk-11-jdk-headless

# Install PySpark and findspark
!pip install pyspark findspark
!pip install pyspark==3.5.0
!pip install pycopy-pkgutil

import pkgutil
print("pyspark installed? ", pkgutil.find_loader("pyspark") is not None)

'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'apt-get' is not recognized as an internal or external command,
operable program or batch file.



[notice] A new release of pip is available: 25.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


pyspark installed?  True



[notice] A new release of pip is available: 25.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: pycopy-pkgutil from https://files.pythonhosted.org/packages/20/15/a16e3ee0fd8ce7acfe37282ae8a8fce52232f81f3f2b5ff64f3e01126e74/pycopy-pkgutil-0.1.1.tar.gz does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
C:\Users\balto\AppData\Local\Temp\ipykernel_25640\4291970874.py:11: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  print("pyspark installed? ", pkgutil.find_loader("pyspark") is not None)


  Using cached pycopy-pkgutil-0.1.1.tar.gz (685 bytes)


In [13]:
# Set Java home and import findspark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

In [14]:
import urllib.request

urls = [
    "https://raw.githubusercontent.com/keeyong/sjsu-data226-FA25/refs/heads/main/week13/data/sample_web_log_1.log.gz",
    "https://raw.githubusercontent.com/keeyong/sjsu-data226-FA25/refs/heads/main/week13/data/sample_web_log_2.log.gz",
    "https://raw.githubusercontent.com/keeyong/sjsu-data226-FA25/refs/heads/main/week13/data/sample_web_log_3.log.gz",
    "https://raw.githubusercontent.com/keeyong/sjsu-data226-FA25/refs/heads/main/week13/data/sample_web_log_4.log.gz",
    "https://raw.githubusercontent.com/keeyong/sjsu-data226-FA25/refs/heads/main/week13/data/sample_web_log_5.log.gz",
    "https://raw.githubusercontent.com/keeyong/sjsu-data226-FA25/refs/heads/main/week13/data/sample_web_log_6.log.gz",
    "https://raw.githubusercontent.com/keeyong/sjsu-data226-FA25/refs/heads/main/week13/data/sample_web_log_7.log.gz"
]

for url in urls:
    filename = url.split("/")[-1]
    print(f"Downloading {filename} ...")
    urllib.request.urlretrieve(url, filename)

print("\nAll log files downloaded!")



All log files downloaded!


## Configure snowflake jar file and set up SparkSession

In [18]:
!cd /usr/local/lib/python3.12/dist-packages/pyspark/jars && wget https://repo1.maven.org/maven2/net/snowflake/snowflake-jdbc/3.19.0/snowflake-jdbc-3.19.0.jar

import pyspark
import os

print(os.path.dirname(pyspark.__file__))

import os

pyspark_path = "/home/balto/airflow_venv/lib/python3.12/site-packages/pyspark"
print(os.listdir(pyspark_path))


c:\Users\balto\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark


The system cannot find the path specified.


FileNotFoundError: [WinError 3] The system cannot find the path specified: '/home/balto/airflow_venv/lib/python3.12/site-packages/pyspark'

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("Logs")
         .master("local[*]")
         .config("spark.ui.enabled", "false")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.driver.memory", "2g")
         .config("spark.executor.memory", "2g")
         .getOrCreate())

spark

KeyboardInterrupt: 

## Create an input dataframe

In [ ]:
import gzip
import shutil
import glob

for gz_file in glob.glob("*.gz"):
    out_file = gz_file.replace(".gz", "")
    print(f"Extracting {gz_file} -> {out_file}")
    with gzip.open(gz_file, "rb") as f_in:
        with open(out_file, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)

df = spark.read.text("sample_web_log_*.log")
df.show(5)

In [ ]:
# Check the number of partitions
print(df.rdd.getNumPartitions())

df.show(truncate=False)

## Create a parsed dataframe (log_df)

In [ ]:
# Extract the necessary information from log data using regular expressions
pattern = r'(\d+\.\d+\.\d+\.\d+) - - \[(.*?)\] "(.*?) (.*?) HTTP.*" (\d+) (\d+)'

log_df = df.select(
    F.regexp_extract("value", pattern, 1).alias("ip"),
    F.regexp_extract("value", pattern, 2).alias("timestamp"),
    F.regexp_extract("value", pattern, 3).alias("method"),
    F.regexp_extract("value", pattern, 4).alias("url"),
    F.regexp_extract("value", pattern, 5).alias("status").cast("integer"),
    F.regexp_extract("value", pattern, 6).alias("size").cast("integer")
)

In [ ]:
log_df.show()

## Let's compute top 404 urls

In [ ]:
# Keep only 404 error logs
error_404_logs = log_df.filter(log_df.status == 404)

In [ ]:
# Group by URL and then count, and sort by count in descending order
url_404_count = error_404_logs.groupBy("url").count().orderBy(F.desc("count"))

In [ ]:
# print the outcome
url_404_count.show()

## Now Let's do this in SparkSQL

In [ ]:
# Register the DataFrame as a temporary SQL table
log_df.createOrReplaceTempView("logs")

In [ ]:
# Use SparkSQL to count URLs with 404 status
url_404_count = spark.sql("""
    SELECT url, COUNT(1) as count
    FROM logs
    WHERE status = 404
    GROUP BY url
    ORDER BY count DESC
""")

In [ ]:
url_404_count.show()

## Let's save this DF (url_404_count) as a table in Snowflake

In [ ]:
from google.colab import userdata

account = userdata.get('snowflake_account')
user = userdata.get('snowflake_userid')
password = userdata.get('snowflake_password')
database = "dev"
schema = "analytics"

url = f"jdbc:snowflake://{account}.snowflakecomputing.com/?db={database}&schema={schema}&user={user}&password={password}"

In [ ]:
url_404_count.write \
    .format("jdbc") \
    .option("driver", "net.snowflake.client.jdbc.SnowflakeDriver") \
    .option("url", url) \
    .mode("overwrite") \
    .option("dbtable", "url_404_count") \
    .save()

In [ ]:
# Now go back to Snowflake and check the table (analytics.url_404_count)